# SHRED on ME4OH full-field OFAM3 SST data

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler

import models  # https://github.com/Jan-Williams/pyshred/blob/main/models.py 
from Data.preprocess_sst_ofam3_shred import create_shred_sequences, load_files, split_ordered_data
from Data.preprocess_sst_ofam3_shred import NA_DATA_PATH
from plotting_functions import plot_sst_map, plot_compare_NA_sst_recon_truth

np.random.seed(42)

# Example visualisation

In [ ]:
date = '1979-01-04'
df = pd.read_csv(f"{NA_DATA_PATH}{date}.csv")
# df['Date'] = pd.to_datetime('1979-01-04')

In [ ]:
df

In [ ]:
plot_sst_map(df, date, min(df.SST), max(df.SST), basin='NA')

# SHRED

In [ ]:
num_sensors = 5
lags = 52

In [ ]:
# Load and reformat data
data, dates, lats, longs = load_files()

In [ ]:
print("Preprocessing data...")
train_data, val_data, test_data, train_dates, val_dates, test_dates = split_ordered_data(data, dates)

# Normalise data
sc = MinMaxScaler()
sc = sc.fit(train_data)  # Computes min and max from training data only (prevent data leakage)
train_transformed = sc.transform(train_data)  
val_transformed = sc.transform(val_data)
test_transformed = sc.transform(test_data)

# Build sequences
sensor_locations = np.random.choice(data.shape[1], size=num_sensors, replace=False)
sensor_coords=(longs[sensor_locations], lats[sensor_locations])
print(f"Sensor location indexes will be: {sensor_locations}")

train_dataset = create_shred_sequences(train_transformed, sensor_locations, num_sensors=num_sensors, lags=lags)
val_dataset = create_shred_sequences(val_transformed, sensor_locations, num_sensors=num_sensors, lags=lags)
test_dataset = create_shred_sequences(test_transformed, sensor_locations, num_sensors=num_sensors, lags=lags)


### STOP HERE if loading saved data

In [ ]:
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# shred = models.SHRED(num_sensors, len(lats), hidden_size=64, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
# validation_errors = models.fit(shred, train_dataset, val_dataset, batch_size=64, num_epochs=1000, lr=1e-3, verbose=True, patience=5)

In [ ]:
# test_recons = sc.inverse_transform(shred(test_dataset.X).detach().cpu().numpy())
# test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())
# print('Test Reconstruction Error: ')
# print(np.linalg.norm(test_recons - test_ground_truth) / np.linalg.norm(test_ground_truth))

In [ ]:
# np.save("recons", test_recons)
# np.save("truth", test_ground_truth)

### START HERE

In [ ]:
test_recons = np.load("recons.npy")
test_ground_truth = np.load("truth.npy")

In [ ]:
for i in np.random.choice(range(len(test_recons)), 1):
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons[i]
    })
    true_df = pd.read_csv(f"{NA_DATA_PATH}{test_dates[i]}.csv")
    print(f"Date: {date}")
    plot_compare_NA_sst_recon_truth(recon_df, true_df, sensor_coords=sensor_coords)

In [ ]:
plot_sst_map(recon_df, test_dates[i], min_sst=min(recon_df.SST), max_sst=max(recon_df.SST), basin='NA', sensor_coords=sensor_coords)

In [ ]:

# plot_sst_map(true_df, date, min(true_df.SST), max(true_df.SST), basin='NA')

### Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

In [ ]:
rmse = np.sqrt(MSE(y_pred=test_recons, y_true=test_ground_truth))
mae = MAE(y_pred=test_recons, y_true=test_ground_truth)
r2 = r2_score(y_pred=test_recons, y_true=test_ground_truth)
print(f"RMSE:\t{rmse:.3f}")
print(f"MAE:\t{mae:.3f}")
print(f"R2:\t{r2:.3f}")

In [ ]:
plt.figure(figsize=(6,6))
plt.grid()
i = 177
plt.scatter(test_ground_truth[i], test_recons[i], alpha=0.25)
plt.plot([-5, 35],
         [-5, 35],
         '--', lw=2, c='r')
plt.xlabel("Observed SST ($\\degree$C)")
plt.ylabel("Reconstructed SST ($\\degree$C)")
plt.title(f"{test_dates[i]}")

In [ ]:
difference = test_recons - test_ground_truth
for i in np.random.choice(range(len(test_recons)), 1):
    diff_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": difference[i]
        })
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons[i]
    })
    true_df = pd.read_csv(f"{NA_DATA_PATH}{test_dates[i]}.csv")
    # plot_sst_map(diff_df, title=f"Difference map for {test_dates[i]}", min_sst=-5, max_sst=5, basin="NA", cmap='bwr')
    print(f"Date: {test_dates[i]}")
    plot_compare_NA_sst_recon_truth(recon_df, true_df, diff_df, sensor_coords=sensor_coords)

### Differences between test reconstructions
Verifying that not just getting identical reconstructions
- Not every point in scatter plot is 0 - so there is difference!

Also see: clear seasonality in the generations being produced!

In [ ]:
avg_diff = []
date1 = []
date2 = []
for i in range(len(test_recons)):
    for j in range(len(test_recons)):
        date1.append(test_dates[i])
        date2.append(test_dates[j])
        avg_diff.append(np.mean(abs(test_recons[i] - test_recons[j])))

diff_df = pd.DataFrame({
    "Date 1": date1,
    "Date 2": date2,
    "Average difference": avg_diff
})

In [ ]:
s = diff_df.plot.scatter(x="Date 1", y="Date 2", c="Average difference", cmap='plasma')
plt.xticks(rotation=90)
plt.grid()